# 🧠 CDM RAG Assistant
**Retrieval-Augmented Generation for Clinical Data Management Documents**

This notebook lets you ask questions about provided CDM/CDS documents and get answers with source citations.

### 📋 How to use this notebook
Run each block **from top to bottom**, in order. You only need to re-run from Block 4 onwards after adding new documents.

| Block | What it does | Run again? |
|-------|-------------|------------|
| 1 | Install packages | Only once |
| 2 | Import libraries | Each session |
| 3 | Load documents (PDF, XLSX, JSON, XML, CSV, DOCX) | When you add files |
| 4 | Chunk text | When you add files |
| 5 | Build search index | When you add files |
| 6 | Connect to Ollama LLM | Each session |
| 7 | Ask a question | Anytime! |
| 8 | Interactive Q&A loop | Anytime! |

## 📦 Block 1 — Install Required Packages
Run this once. It installs everything the notebook needs.

> ⚠️ **Before running Block 6**, you also need to install Ollama separately:
> 1. Go to [https://ollama.com](https://ollama.com) and download the app
> 2. Open Terminal and run: `ollama pull mistral`
> 3. Ollama will then run in the background automatically

In [1]:
# Install all required Python packages
# Run this once — it installs everything the notebook needs
import sys

!{sys.executable} -m pip install --upgrade pip --quiet
!{sys.executable} -m pip install sentence-transformers chromadb pypdf openpyxl python-docx python-pptx requests tqdm --quiet
!{sys.executable} -m pip install --upgrade jupyter ipywidgets --quiet

print("✅ All packages installed!")


✅ All packages installed!


## 📚 Block 2 — Import Libraries
This loads all the tools we need into memory. Run this at the start of every session.

In [2]:
import os
import json
import csv
import textwrap
import xml.etree.ElementTree as ET
import requests
import numpy as np
import chromadb

from pypdf import PdfReader
from pptx import Presentation
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
import openpyxl
import docx  # python-docx

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


## 📁 Block 3 — Load Documents

**👇 Only change this line:** set `DOCS_FOLDER` to the path of your documents folder.

Supported file types:
- 📄 **PDF** — extracts text page by page
- 📊 **XLSX** — reads each sheet row by row
- 📝 **DOCX** — reads paragraphs
- 🗂️ **JSON** — reads structured data
- 🔖 **XML** — reads element text
- 📋 **CSV** — reads rows as text
- 📃 **TXT** — reads plain text
-    **PPTX** - reads each slide and its speaker notes

In [3]:
# ============================================================
# 🔴 CHANGE THIS to your documents folder path
# ============================================================
DOCS_FOLDER = "../data/raw/"
# ============================================================
#
# 📁 Document authority is set by subfolder:
#
#   data/raw/regulatory/   → binding requirements (ICH, FDA, EMA)
#   data/raw/opinion/      → position papers, white papers, SCDM/JSCDM
#
# Drop a file into the correct subfolder — no other configuration needed.
# Files found outside these two subfolders will raise an error.
# ============================================================

AUTHORITY_FOLDERS = {
    "regulatory": "regulatory",
    "opinion":    "opinion",
}


def get_authority(filepath):
    """Derive authority level from the subfolder the file lives in.

    Raises a clear error if the file is not inside a recognised subfolder,
    so nothing is ever classified by accident.
    """
    # Normalise path separators and split into parts
    parts = filepath.replace("\\", "/").split("/")
    for part in parts:
        if part.lower() in AUTHORITY_FOLDERS:
            return AUTHORITY_FOLDERS[part.lower()]
    raise ValueError(
        f"\n❌ Cannot classify: '{os.path.basename(filepath)}'\n"
        f"   Move it into one of these subfolders before loading:\n"
        f"     {DOCS_FOLDER}regulatory/   ← binding requirements (ICH, FDA, EMA)\n"
        f"     {DOCS_FOLDER}opinion/       ← position papers, white papers"
    )


# --- Helper: detect heading from PDF page text ---
def extract_heading(text):
    """Try to find a heading on the page (short ALL-CAPS line)."""
    for line in text.split("\n"):
        stripped = line.strip()
        if 4 < len(stripped) < 80 and stripped.isupper():
            return stripped
    return "Unknown"


# --- Loader functions per file type ---
def load_pdf(filepath):
    """Load a PDF file. Returns one entry per page."""
    docs = []
    filename  = os.path.basename(filepath)
    authority = get_authority(filepath)
    try:
        reader = PdfReader(filepath)
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                docs.append({
                    "text":      text.strip(),
                    "source":    filename,
                    "page":      i + 1,
                    "heading":   extract_heading(text),
                    "authority": authority,
                })
    except Exception as e:
        print(f"  ❌ Error reading PDF {filename}: {e}")
    return docs


def load_xlsx(filepath):
    """Load an Excel file. Returns one entry per sheet (rows joined as text)."""
    docs = []
    filename  = os.path.basename(filepath)
    authority = get_authority(filepath)
    try:
        wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            rows_text = []
            for row in ws.iter_rows(values_only=True):
                row_str = " | ".join(str(cell) for cell in row if cell is not None)
                if row_str.strip():
                    rows_text.append(row_str)
            if rows_text:
                docs.append({
                    "text":      "\n".join(rows_text),
                    "source":    filename,
                    "page":      sheet_name,
                    "heading":   f"Sheet: {sheet_name}",
                    "authority": authority,
                })
    except Exception as e:
        print(f"  ❌ Error reading XLSX {filename}: {e}")
    return docs


def load_docx(filepath):
    """Load a Word document. Returns one entry per document."""
    docs = []
    filename  = os.path.basename(filepath)
    authority = get_authority(filepath)
    try:
        doc = docx.Document(filepath)
        paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        full_text  = "\n".join(paragraphs)
        if full_text:
            docs.append({
                "text":      full_text,
                "source":    filename,
                "page":      1,
                "heading":   paragraphs[0][:80] if paragraphs else "Unknown",
                "authority": authority,
            })
    except Exception as e:
        print(f"  ❌ Error reading DOCX {filename}: {e}")
    return docs


def load_pptx(filepath):
    """Load a PowerPoint file. Extracts text from slides and speaker notes."""
    docs = []
    filename  = os.path.basename(filepath)
    authority = get_authority(filepath)
    try:
        prs = Presentation(filepath)
        for i, slide in enumerate(prs.slides, 1):
            parts = []
            for shape in slide.shapes:
                if shape.has_text_frame:
                    parts.append(shape.text_frame.text.strip())
            if slide.has_notes_slide:
                notes = slide.notes_slide.notes_text_frame.text.strip()
                if notes:
                    parts.append(f"[Speaker notes] {notes}")
            text = "\n".join(p for p in parts if p)
            if text:
                docs.append({
                    "text":      text,
                    "source":    filename,
                    "page":      i,
                    "heading":   f"Slide {i}",
                    "authority": authority,
                })
    except Exception as e:
        print(f"  ❌ Error reading PPTX {filename}: {e}")
    return docs


def load_json(filepath):
    """Load a JSON file. Converts to text."""
    docs = []
    filename  = os.path.basename(filepath)
    authority = get_authority(filepath)
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        text = json.dumps(data, indent=2, ensure_ascii=False)
        docs.append({
            "text":      text,
            "source":    filename,
            "page":      1,
            "heading":   "JSON Document",
            "authority": authority,
        })
    except Exception as e:
        print(f"  ❌ Error reading JSON {filename}: {e}")
    return docs


def load_xml(filepath):
    """Load an XML file. Extracts all text content."""
    docs = []
    filename  = os.path.basename(filepath)
    authority = get_authority(filepath)
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        texts = []
        for elem in root.iter():
            if elem.text and elem.text.strip():
                texts.append(f"{elem.tag}: {elem.text.strip()}")
        full_text = "\n".join(texts)
        if full_text:
            docs.append({
                "text":      full_text,
                "source":    filename,
                "page":      1,
                "heading":   f"XML Root: {root.tag}",
                "authority": authority,
            })
    except Exception as e:
        print(f"  ❌ Error reading XML {filename}: {e}")
    return docs


def load_csv(filepath):
    """Load a CSV file. Returns one entry for the whole file."""
    docs = []
    filename  = os.path.basename(filepath)
    authority = get_authority(filepath)
    try:
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            reader = csv.reader(f)
            rows = [" | ".join(row) for row in reader if any(cell.strip() for cell in row)]
        full_text = "\n".join(rows)
        if full_text:
            docs.append({
                "text":      full_text,
                "source":    filename,
                "page":      1,
                "heading":   "CSV Document",
                "authority": authority,
            })
    except Exception as e:
        print(f"  ❌ Error reading CSV {filename}: {e}")
    return docs


def load_txt(filepath):
    """Load a plain text file."""
    docs = []
    filename  = os.path.basename(filepath)
    authority = get_authority(filepath)
    try:
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            text = f.read().strip()
        if text:
            docs.append({
                "text":      text,
                "source":    filename,
                "page":      1,
                "heading":   "Text Document",
                "authority": authority,
            })
    except Exception as e:
        print(f"  ❌ Error reading TXT {filename}: {e}")
    return docs


# --- Main loader: scans folder and picks the right loader ---
LOADERS = {
    ".pdf":  load_pdf,
    ".xlsx": load_xlsx,
    ".xls":  load_xlsx,
    ".docx": load_docx,
    ".pptx": load_pptx,
    ".json": load_json,
    ".xml":  load_xml,
    ".csv":  load_csv,
    ".txt":  load_txt,
}


def load_all_documents(folder_path):
    """Walk a folder and load all supported file types."""
    all_docs = []
    skipped  = []

    if not os.path.exists(folder_path):
        print(f"❌ Folder not found: {folder_path}")
        print("   Please update DOCS_FOLDER above to your actual documents folder.")
        return all_docs

    for root, dirs, files in os.walk(folder_path):
        for filename in sorted(files):
            ext = os.path.splitext(filename)[1].lower()

            # Warn clearly about old .ppt format — not supported
            if ext == ".ppt":
                print(f"  ⚠️  Skipped (old format): {filename}")
                print(f"     → Open in PowerPoint and save as .pptx, then re-add.")
                skipped.append(filename)
                continue

            if ext in LOADERS:
                full_path = os.path.join(root, filename)
                try:
                    print(f"  📄 Loading: {filename}")
                    docs = LOADERS[ext](full_path)
                    print(f"     → {len(docs)} section(s) found  [{get_authority(full_path).upper()}]")
                    all_docs.extend(docs)
                except ValueError as e:
                    print(e)
                    print("\n⛔ Loading stopped. Classify the file above and re-run Block 3.")
                    return all_docs
            elif not filename.startswith("."):
                skipped.append(filename)

    print(f"\n✅ Total sections loaded: {len(all_docs)}")
    if skipped:
        print(f"⚠️  Skipped: {chr(10).join(skipped)}")
    return all_docs


# Run the loader
print(f"🔍 Scanning folder: {DOCS_FOLDER}\n")
docs = load_all_documents(DOCS_FOLDER)


🔍 Scanning folder: ../data/raw/

  📄 Loading: Guidance for eCOA Development in Clinical Trials.pdf
     → 23 section(s) found  [OPINION]
  📄 Loading: jscdm-116-lebedys.pdf
     → 20 section(s) found  [OPINION]
  📄 Loading: jscdm-117-hills.pdf
     → 12 section(s) found  [OPINION]
  📄 Loading: jscdm-118-amatya.pdf
     → 13 section(s) found  [OPINION]
  📄 Loading: jscdm-20-stokman.pdf
     → 8 section(s) found  [OPINION]
  📄 Loading: jscdm-260-king.pdf
     → 9 section(s) found  [OPINION]
  📄 Loading: jscdm-262-king.pdf
     → 10 section(s) found  [OPINION]
  📄 Loading: jscdm-263-king.pdf
     → 10 section(s) found  [OPINION]
  📄 Loading: jscdm-264-king.pdf
     → 7 section(s) found  [OPINION]
  📄 Loading: jscdm-29-pestronk.pdf
     → 19 section(s) found  [OPINION]
  📄 Loading: jscdm-30-eade.pdf
     → 24 section(s) found  [OPINION]
  📄 Loading: jscdm-31-montano.pdf
     → 22 section(s) found  [OPINION]
  📄 Loading: jscdm-411-famatiga-fay.pdf
     → 21 section(s) found  [OPINION]
  📄 Lo

## ✂️ Block 4 — Split Text into Chunks

Long documents are split into smaller overlapping pieces so the search engine can find precise answers.

- `CHUNK_SIZE` = how many words per chunk (500 is a good default)
- `CHUNK_OVERLAP` = how many words overlap between chunks (helps avoid cutting answers in half)

In [4]:
# ============================================================
# Settings — you can leave these as-is to start
# ============================================================
CHUNK_SIZE    = 500   # words per chunk
CHUNK_OVERLAP = 50    # words of overlap between chunks
# ============================================================

chunks   = []  # stores the text of each chunk
metadata = []  # stores source, page, heading, authority for each chunk

for doc in docs:
    words = doc["text"].split()
    step  = CHUNK_SIZE - CHUNK_OVERLAP

    for i in range(0, len(words), step):
        chunk_words = words[i : i + CHUNK_SIZE]
        chunk_text  = " ".join(chunk_words)
        if chunk_text.strip():
            chunks.append(chunk_text)
            metadata.append({
                "source":    doc["source"],
                "page":      doc["page"],
                "heading":   doc["heading"],
                "authority": doc["authority"],
            })

print(f"✅ Created {len(chunks)} chunks from {len(docs)} document sections")
print(f"   (chunk size: {CHUNK_SIZE} words, overlap: {CHUNK_OVERLAP} words)")


✅ Created 2058 chunks from 1611 document sections
   (chunk size: 500 words, overlap: 50 words)


## 🧠 Block 5 — Build the Search Index

This converts all your text chunks into numbers (called *embeddings*) that the computer can compare mathematically. Then it stores them in a fast search index.

**This may take a minute or two** — it's doing a lot of maths!

The model `all-MiniLM-L6-v2` is small, fast, and works well for document search.

In [5]:
import torch

# Step 1: Load the embedding model
print("⏳ Loading embedding model (downloads once, then cached)...")

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"   Using device: {device}")

embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("✅ Embedding model ready!")

# Step 2: Encode all chunks into embeddings
print(f"\n⏳ Encoding {len(chunks)} chunks into embeddings...")
embeddings = embed_model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True,
)
embeddings = embeddings.astype("float32")
print(f"✅ Embeddings created: {embeddings.shape} (chunks × dimensions)")

# Step 3: Build the search index
print("\n⏳ Building search index...")
chroma_client = chromadb.Client()

# Reset collection if re-running this block
try:
    chroma_client.delete_collection("rag_docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="rag_docs",
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    ids        = [str(i) for i in range(len(chunks))],
    embeddings = embeddings.tolist(),
    documents  = chunks,
    metadatas  = metadata,
)

print(f"✅ Search index ready with {collection.count()} vectors")


⏳ Loading embedding model (downloads once, then cached)...
   Using device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready!

⏳ Encoding 2058 chunks into embeddings...


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

✅ Embeddings created: (2058, 384) (chunks × dimensions)

⏳ Building search index...
✅ Search index ready with 2058 vectors


## 🤖 Block 6 — Connect to Ollama (Local AI)

This connects to Ollama running on your Mac. Ollama must be installed and running before this works.

**Quick setup (only needed once):**
1. Download Ollama from [https://ollama.com](https://ollama.com)
2. Open Terminal and run: `ollama pull mistral`
3. Ollama runs automatically in the background

**Which model to use?**
- `mistral` — best balance of speed and quality (recommended ✅)
- `llama3.2` — also great, slightly slower
- `phi3` — fastest, good for quick tests

In [6]:
# ============================================================
# Settings
# ============================================================
OLLAMA_MODEL = "mistral"         # Change to llama3.2 or phi3 if preferred
OLLAMA_URL   = "http://localhost:11434/api/generate"
# ============================================================


def ask_ollama(prompt, model=OLLAMA_MODEL, temperature=0.3):
    """
    Send a prompt to Ollama and return the response text.

    Args:
        prompt (str): The full prompt to send
        model (str): Which Ollama model to use
        temperature (float): How creative/random the answer is (0 = factual, 1 = creative)
    Returns:
        str: The LLM's answer
    """
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model":  model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": temperature}
            },
            timeout=120  # wait up to 2 minutes for a response
        )
        response.raise_for_status()
        return response.json().get("response", "").strip()

    except requests.exceptions.ConnectionError:
        return (
            "❌ Cannot connect to Ollama.\n"
            "   Make sure Ollama is installed and running.\n"
            "   Download from: https://ollama.com\n"
            f"  Then in Terminal run: ollama pull {OLLAMA_MODEL}"
        )
    except Exception as e:
        return f"❌ Error: {e}"


# Test the connection
print(f"🔗 Testing connection to Ollama (model: {OLLAMA_MODEL})...")
test_reply = ask_ollama("Reply with exactly: 'Ollama is working!'")
print(f"Response: {test_reply}")

if "working" in test_reply.lower() or "ollama" in test_reply.lower():
    print("\n✅ Ollama is connected and ready!")
else:
    print("\n⚠️  Got a response — Ollama seems to be running.")

🔗 Testing connection to Ollama (model: mistral)...
Response: Ollama is working!

✅ Ollama is connected and ready!


## 🔍 Block 7 — Search + Ask a Question

This is the core of the RAG system:
1. Your question is converted to an embedding
2. FAISS finds the most relevant chunks from your documents
3. Those chunks are sent to Ollama as context
4. Ollama answers your question and cites the sources

**👇 Change `MY_QUESTION` to whatever you want to ask!**

In [7]:
# ============================================================
# 👇 Type your question here
# ============================================================
MY_QUESTION = "What skills does a Clinical Data Manager need to transition to Data Science?"
TOP_K       = 5   # How many document chunks to retrieve (3–7 is usually best)
# ============================================================


# ANSI colour helpers — work in Jupyter and most terminals
def _c(code, text): return f"\033[{code}m{text}\033[0m"
def green(t):    return _c('32;1', t)
def yellow(t):   return _c('33;1', t)
def red(t):      return _c('31;1', t)
def magenta(t):  return _c('35;1', t)
def blue(t):     return _c('34;1', t)
def bold(t):     return _c('1',    t)
def dim(t):      return _c('2',    t)


def search_documents(question, top_k=5):
    """Search index, then re-rank: regulatory sources first, then by relevance."""
    query_embedding = embed_model.encode([question]).astype("float32")
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
    )

    output = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        output.append({
            "text":      results["documents"][0][i],
            "source":    meta.get("source", "unknown"),
            "page":      meta.get("page", "?"),
            "heading":   meta.get("heading", "?"),
            "authority": meta.get("authority", "opinion"),
            "distance":  float(results["distances"][0][i]),
        })

    # Regulatory sources surface first; within each group sort by relevance
    output.sort(key=lambda x: (
        0 if x["authority"] == "regulatory" else 1,
        x["distance"],
    ))
    return output


def build_prompt(question, retrieved_chunks):
    """Build the LLM prompt, labelling each source with its authority level."""
    seen, unique = set(), []
    for chunk in retrieved_chunks:
        key = (chunk["source"], chunk["page"])
        if key not in seen:
            seen.add(key)
            unique.append(chunk)

    context_parts = []
    for i, chunk in enumerate(unique, 1):
        fname     = os.path.basename(chunk["source"])
        authority = chunk["authority"].upper()
        context_parts.append(
            f"[{i}] [{authority}] {fname} — page {chunk['page']}\n"
            f"{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    return f"""You are an expert assistant in Clinical Data Management (CDM) and Clinical Data Science (CDS).

Answer the question below using ONLY the document excerpts provided.

IMPORTANT — apply these language rules strictly based on source type:
- [REGULATORY] sources represent binding requirements from bodies such as ICH, FDA, or EMA.
  Use language such as: must, is required to, mandates, is prohibited, shall.
- [OPINION] sources represent professional recommendations and position papers.
  Use language such as: recommends, suggests, proposes, considers, advises.
If a REGULATORY source conflicts with an OPINION source on the same point, flag this explicitly.
Always cite sources as: (Source: <filename>, Page: <page>)
Do not make up information.

=== DOCUMENT CONTEXT ===
{context}

=== QUESTION ===
{question}

=== ANSWER ==="""


def ask_question(question, top_k=5):
    """Full RAG pipeline: retrieve → re-rank → deduplicate → prompt → answer."""
    print()
    print(bold(f"🔍 Searching for: \"{question}\""))

    retrieved = search_documents(question, top_k=top_k)
    print(dim(f"   {len(retrieved)} chunks retrieved, regulatory sources ranked first"))

    prompt = build_prompt(question, retrieved)
    print(dim(f"   Asking {OLLAMA_MODEL}..."))
    answer = ask_ollama(prompt)

    # ── Answer (wrapped for readability) ─────────────────────────────────
    print("\n" + bold("=" * 60))
    print(bold("💬  ANSWER"))
    print(bold("=" * 60))
    for line in answer.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=80))
        else:
            print()

    # ── Sources (deduplicated, regulatory first) ──────────────────────────
    seen, printed = set(), []
    for r in retrieved:
        key = (r["source"], r["page"])
        if key not in seen:
            seen.add(key)
            printed.append(r)

    print("\n" + bold("-" * 60))
    print(bold("📚  SOURCES  ") + dim("(regulatory first, then by relevance)"))
    print(bold("-" * 60))

    for i, r in enumerate(printed, 1):
        d         = r["distance"]
        authority = r["authority"]

        auth_badge = magenta("⚖️  REGULATORY") if authority == "regulatory" else blue("💬  OPINION   ")

        if d < 0.3:
            rel_badge = green("🟢 HIGH  ")
        elif d < 0.6:
            rel_badge = yellow("🟡 MEDIUM")
        else:
            rel_badge = red("🔴 LOW   ")

        fname = os.path.basename(r["source"])
        print(f"  {auth_badge}  {rel_badge}  [{i}] {bold(fname)}  —  page {r['page']}")
        print(dim(f"         relevance score: {d:.3f}  (lower = closer match)"))
        print()

    return answer


# Run!
answer = ask_question(MY_QUESTION, top_k=TOP_K)



🔍 Searching for: "What skills does a Clinical Data Manager need to transition to Data Science?"
   5 chunks retrieved, regulatory sources ranked first
   Asking mistral...

💬  ANSWER
A Clinical Data Manager needs to build on the core CDM skillsets and focus on
emerging opportunities offered by technology, regulations, and Clinical
Development strategies. The following are some of the emerging skillsets that a
Clinical Data Manager should consider acquiring to transition to Data Science:

1. Robust critical thinking and process knowledge (Source: 2019_Evolution-of-
CDM-to-CDS-Part-1-Drivers.pdf, page 20)
2. Broader cross functional collaboration (Source: 2019_Evolution-of-CDM-to-CDS-
Part-1-Drivers.pdf, page 20)
3. Ability to align the flows of data with the need of the next generation
clinical protocol (Source: 2019_Evolution-of-CDM-to-CDS-Part-1-Drivers.pdf, page
20)
4. Deep knowledge of data including the characteristics of different types of
data (Source: 2019_Evolution-of-CDM-to-C

## 💬 Block 8 — Interactive Q&A Loop

This lets you ask multiple questions without re-running cells.
Type your question and press Enter. Type `quit` or `exit` to stop.

In [8]:
print("💬 Interactive Q&A — type your question and press Enter")
print("   Type 'quit' or 'exit' to stop\n")

while True:
    question = input("\n❓ Your question: ").strip()

    if not question:
        print("   ⚠️  Please type a question.")
        continue

    if question.lower() in ("quit", "exit", "stop", "q"):
        print("\n👋 Goodbye! Come back when you have more questions.")
        break

    ask_question(question, top_k=5)

💬 Interactive Q&A — type your question and press Enter
   Type 'quit' or 'exit' to stop


🔍 Searching for: "What are the requirements for data collection?"
   5 chunks retrieved, regulatory sources ranked first
   Asking mistral...

💬  ANSWER
The requirements for data collection, as per the provided document excerpts, are
as follows:

1. [OPINION] jscdm-411-famatiga-fay.pdf — page 2 (GCDMP)
   - Adherence to regulatory guidelines and applicable standards
   - Data Identification (protocol review)
   - CRF Design Practices (formatting of how data should be collected, managing
conditional logic, minimizing data redundancies, managing data privacy, handling
language translations)
   - Development Considerations (data mapping, edit checks, and data/system
integration)
   - Libraries and Standards (data element libraries and metadata, standards)
   - Change Control and Versioning

2. [OPINION] jscdm-411-famatiga-fay.pdf — page 3 (GCDMP)
   - Data should be collected and maintained in a secu

---
## 📖 Quick Reference

### Understanding relevance scores
Sources are sorted from most to least relevant. The score is a cosine distance (0 = perfect match, 1 = unrelated):
- 🟢 **Under 0.3** — highly relevant
- 🟡 **0.3 – 0.6** — somewhat relevant
- 🔴 **Over 0.6** — weak match (may not add useful context)

### Tips for better answers
- Ask specific questions — *"What does Part 3 say about the CDM role evolution?"* works better than *"tell me about CDM"*
- If the answer seems off, try rephrasing your question
- Increase `TOP_K` to 7 or 8 if answers feel incomplete
- Decrease `CHUNK_SIZE` to 300 if you want more precise source citations
